<div style="
  border:3px solid #0F5132;      /* azul oscuro (borde) */
  border-radius:10px;
  background-color:rgba(209,231,221,0.7);      /* azul claro (fondo) */
  padding:4px 8px;              /* menos espacio interno */
  overflow:auto;">
<h2 align="left" style="color:#0F5132;margin:0;"><b>Importación de librerías y funciones para el pipeline</b></h2>
</div>


In [8]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import yeojohnson as _yj


<div style="
  border:3px solid #0F5132;      /* azul oscuro (borde) */
  border-radius:10px;
  background-color:rgba(209,231,221,0.7);      /* azul claro (fondo) */
  padding:4px 8px;              /* menos espacio interno */
  overflow:auto;">
<h2 align="left" style="color:#0F5132;margin:0;"><b>Preparación de los dataframe train</b></h2>
</div>


In [9]:
#importación del Train
DATA_PATH = Path("data/train.xlsx")  # 
df_train = pd.read_excel(DATA_PATH)

# importación del Test
# DATA_PATH_TEST = Path("data/test.xlsx")
# df_test = pd.read_excel(DATA_PATH_TEST)
   

# Separación objetivo / predictoras (ajusta 'deseada' si tu target tiene otro nombre)
y_train = df_train["deseada"].copy()
x_train = df_train.drop(columns=["deseada"]).copy()

<div style="
  border:3px solid #0F5132;      /* azul oscuro (borde) */
  border-radius:10px;
  background-color:rgba(209,231,221,0.7);      /* azul claro (fondo) */
  padding:4px 8px;              /* menos espacio interno */
  overflow:auto;">
<h2 align="left" style="color:#0F5132;margin:0;"><b>Pipeline y entrenamiento modelo final</b></h2>
</div>

<div style="
  border:3px solid #0F5132;      /* azul oscuro (borde) */
  border-radius:10px;
  background-color:rgba(209,231,221,0.7);      /* azul claro (fondo) */
  padding:4px 8px;              /* menos espacio interno */
  overflow:auto;">
<h3 align="left" style="color:#0F5132;margin:0;"><b>Preparación del dataset eliminando x9 y preparando x8</b></h3>
</div>

In [10]:
MODEL_PATH_DISCOVERY = "model/hgb_esc4_pipeline_discovery.joblib"
MODEL_PATH_FINAL     = "model/hgb_esc4_pipeline_final.joblib"


# ----- Helpers -----

def preparar_x8_como_etiqueta(serie):
    s_num = pd.to_numeric(serie.astype(str).str.extract(r'(\d+)', expand=False), errors='coerce')
    s_int = s_num.astype('Int64')
    return s_int.astype('string').fillna('Missing')

def elimina_x9_df(X: pd.DataFrame) -> pd.DataFrame:
    """Elimina x9 si existe y devuelve DataFrame (mantener nombres)."""
    return X.drop(columns=['x9'], errors='ignore')

def limpiar_x8_to_label(X):
    """
    Recibe X[:,0] con 'x8' y devuelve array 2D (n,1) de etiquetas string:
      - extrae números (admite '28', '28.0', '28 días', ...)
      - convierte a Int 'nullable' y etiqueta faltantes como 'Missing'
    """
    s = pd.Series(np.asarray(X).ravel(), dtype="object")
    if "preparar_x8_como_etiqueta" in globals():
        lbl = preparar_x8_como_etiqueta(s)
    else:
        s_num = pd.to_numeric(s.astype(str).str.extract(r'(\d+)', expand=False), errors='coerce')
        s_int = s_num.astype('Int64')
        lbl = s_int.astype('string').fillna('Missing')
    return lbl.to_numpy().reshape(-1, 1)

drop_x9_step  = FunctionTransformer(elimina_x9_df, validate=False)
clean_x8_step = FunctionTransformer(limpiar_x8_to_label, validate=False)

<div style="
  border:3px solid #0F5132;      /* azul oscuro (borde) */
  border-radius:10px;
  background-color:rgba(209,231,221,0.7);      /* azul claro (fondo) */
  padding:4px 8px;              /* menos espacio interno */
  overflow:auto;">
<h3 align="left" style="color:#0F5132;margin:0;"><b>Transformación Yeojohnson</b></h3>
</div>

In [11]:
class YeoJohnsonLearn(BaseEstimator, TransformerMixin):
    """Aprende λ por columna en fit y aplica Yeo-Johnson con esos λ en transform."""
    def __init__(self, col_order=None):
        self.col_order = col_order        # ← no mutar aquí
        self.lambdas_  = None

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        # Si necesitas lista, hazlo ahora:
        _ = list(self.col_order) if self.col_order is not None else None

        self.lambdas_ = np.empty(X.shape[1], dtype=float)
        was = plt.isinteractive()
        try:
            plt.ioff()
            for j in range(X.shape[1]):
                _, lam = _yj(X[:, j])     # aprende λ con el TRAIN que reciba
                self.lambdas_[j] = float(lam)
        finally:
            plt.close('all')
            if was: plt.ion()
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        Z = np.empty_like(X, dtype=float)
        for j in range(X.shape[1]):
            Z[:, j] = _yj(X[:, j], lmbda=self.lambdas_[j])
        return Z

<div style="
  border:3px solid #0F5132;      /* azul oscuro (borde) */
  border-radius:10px;
  background-color:rgba(209,231,221,0.7);      /* azul claro (fondo) */
  padding:4px 8px;              /* menos espacio interno */
  overflow:auto;">
<h3 align="left" style="color:#0F5132;margin:0;"><b>Pasos para la transformación instanciados:</b></h3>
</div>

In [12]:
imp_mediana = SimpleImputer(strategy='median')
yj_learn    = YeoJohnsonLearn(col_order=['x1','x2','x3','x5'])
std_scaler  = StandardScaler()


#Crear dummies para x8
ohe_x8 = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)

<div style="
  border:3px solid #0F5132;      /* azul oscuro (borde) */
  border-radius:10px;
  background-color:rgba(209,231,221,0.7);      /* azul claro (fondo) */
  padding:4px 8px;              /* menos espacio interno */
  overflow:auto;">
<h3 align="left" style="color:#0F5132;margin:0;"><b>Transformación hasta conseguir condiciones de escenario4 de las pruebas</b></h3>
</div>

In [13]:
num_yj  = ['x1','x2','x3','x5']
num_pas = ['x4','x6','x7','x10']
cat_x8  = ['x8']

preprocesa = ColumnTransformer([
    ('yj',  Pipeline([('imp', imp_mediana),
                      ('yj',  yj_learn),
                      ('sc',  std_scaler)]), num_yj),

    ('num', Pipeline([('imp', imp_mediana),
                      ('sc',  std_scaler)]),  num_pas),

    ('x8',  Pipeline([('clean', clean_x8_step),
                      ('ohe',   ohe_x8)]),    cat_x8),
], remainder='drop')

<div style="
  border:3px solid #0F5132;      /* azul oscuro (borde) */
  border-radius:10px;
  background-color:rgba(209,231,221,0.7);      /* azul claro (fondo) */
  padding:4px 8px;              /* menos espacio interno */
  overflow:auto;">
<h3 align="left" style="color:#0F5132;margin:0;"><b>Discovery (early stopping para hallar n_iter_ óptimo)</b></h3>
</div>

In [18]:
modelo_discovery = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_iter=2000,
    early_stopping=True,
    validation_fraction=0.2,
    n_iter_no_change=100,
    random_state=42
)

pipe_discovery = Pipeline([
    ('elimina_x9', drop_x9_step),
    ('preprocesa', preprocesa),
    ('modelo',     modelo_discovery)
], verbose=True)

pipe_discovery.fit(x_train, y_train)
joblib.dump(pipe_discovery, MODEL_PATH_DISCOVERY)
best_iter = int(pipe_discovery.named_steps['modelo'].n_iter_)
print(f"🔎 n_iter_ óptimo (early stopping): {best_iter}")

# Trazabilidad: λ aprendidos en esta fase
yj_used_disc = pipe_discovery.named_steps['preprocesa']\
                             .named_transformers_['yj']\
                             .named_steps['yj']\
                             .lambdas_
print("λ discovery (x1,x2,x3,x5):", yj_used_disc)


[Pipeline] ........ (step 1 of 3) Processing elimina_x9, total=   0.0s
[Pipeline] ........ (step 2 of 3) Processing preprocesa, total=   0.0s
[Pipeline] ............ (step 3 of 3) Processing modelo, total=   0.6s
🔎 n_iter_ óptimo (early stopping): 352
λ discovery (x1,x2,x3,x5): [ 0.10918841  0.02848669 -1.22706494  0.3227266 ]


<div style="
  border:3px solid #0F5132;      /* azul oscuro (borde) */
  border-radius:10px;
  background-color:rgba(209,231,221,0.7);      /* azul claro (fondo) */
  padding:4px 8px;              /* menos espacio interno */
  overflow:auto;">
<h3 align="left" style="color:#0F5132;margin:0;"><b>Entrenamiento del modelo final</b></h3>
</div>

In [20]:
modelo_final = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_iter=best_iter,
    early_stopping=False,
    random_state=42
)

# reconstruimos el preprocesado para volver a aprender medianas/λ/escalas con TODO el train
preprocesa_final = ColumnTransformer([
    ('yj',  Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('yj',  YeoJohnsonLearn(col_order=num_yj)),
                      ('sc',  StandardScaler())]), num_yj),

    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc',  StandardScaler())]),  num_pas),

    ('x8',  Pipeline([('clean', FunctionTransformer(limpiar_x8_to_label, validate=False)),
                      ('ohe',   ohe_x8)]),          cat_x8),
], remainder='drop')

pipe_final = Pipeline([
    ('elimina_x9', FunctionTransformer(elimina_x9_df, validate=False)),
    ('preprocesa', preprocesa_final),
    ('modelo',     modelo_final)
], verbose=True)

pipe_final.fit(x_train, y_train)
joblib.dump(pipe_final, MODEL_PATH_FINAL)
print(f"✅ Modelo FINAL guardado en: {MODEL_PATH_FINAL}")

# λ definitivos (aprendidos con TODO el train)
yj_used_final = pipe_final.named_steps['preprocesa']\
                          .named_transformers_['yj']\
                          .named_steps['yj']\
                          .lambdas_
print("λ FINAL (x1,x2,x3,x5):", yj_used_final)

[Pipeline] ........ (step 1 of 3) Processing elimina_x9, total=   0.0s
[Pipeline] ........ (step 2 of 3) Processing preprocesa, total=   0.0s
[Pipeline] ............ (step 3 of 3) Processing modelo, total=   0.8s
✅ Modelo FINAL guardado en: model/hgb_esc4_pipeline_final.joblib
λ FINAL (x1,x2,x3,x5): [ 0.10918841  0.02848669 -1.22706494  0.3227266 ]


<div style="
  border:3px solid #D4AF37;                 /* dorado */
  border-radius:12px;
  background:linear-gradient(135deg, rgba(17,17,17,0.96), rgba(28,28,28,0.92));  /* grafito */
  padding:8px 12px;
  box-shadow:0 6px 16px rgba(0,0,0,0.25), inset 0 1px 0 rgba(255,255,255,0.04);
  overflow:auto;">
  <h3 align="left" style="color:#F5D27A;margin:0;letter-spacing:0.3px;"><b>
    Predicción
  </b></h3>
</div>


In [ ]:
# ==========================================
# Predicción oficial sobre test.xlsx
# ==========================================
import pandas as pd
import joblib
from pathlib import Path

def predict_test(test_df: pd.DataFrame, model_path: str = "hgb_esc4_pipeline_final.joblib"):
    """Aplica el pipeline FINAL al test crudo y devuelve predicciones."""
    pipe = joblib.load(model_path)
    return pipe.predict(test_df)

# Ejemplo de uso cuando llegue el fichero de la competición:
# TEST_PATH = Path("data/test.xlsx")
# test_df = pd.read_excel(TEST_PATH)
# y_pred  = predict_test(test_df)
# pd.DataFrame({"pred": y_pred}).to_csv("submission.csv", index=False)
# print("✅ submission.csv creado")
